# 00 · Run everything
Data engineering → feature engineering → model training → evaluation, end to end. For a quick test set `start_date` to e.g. 2026-04-01.

Settings come from `conf/parameters.yml`; the widgets below override the `run` section. Outputs are listed in `conf/catalog.yml`. If a node fails, fix it and set `from_node` to resume.

In [0]:
%pip install -q pyyaml lightgbm
dbutils.library.restartPython()

In [0]:
dbutils.widgets.text("output_schema", "ewec_dev_powerops.scratch_yash", "Output schema")
dbutils.widgets.text("product", "dayahead_15min", "Product")
dbutils.widgets.text("start_date", "2024-01-23", "First origin date")
dbutils.widgets.text("end_date", "", "Last origin date (blank = yesterday)")
dbutils.widgets.text("from_node", "", "Resume from node (blank = start)")

In [0]:
import os, sys
REPO = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(REPO, "src"))
for m in [m for m in list(sys.modules) if m.startswith("uc2_solar")]:
    del sys.modules[m]          # reload code after a git pull
from uc2_solar.runner import run
from uc2_solar.registry import register_pipelines

OVERRIDES = {k: dbutils.widgets.get(k) for k in ["output_schema", "product", "start_date", "end_date"]}
FROM_NODE = dbutils.widgets.get("from_node").strip() or None
print("Repo:", REPO)

## Pipeline
The nodes that will run, with their inputs and outputs:

In [0]:
print(register_pipelines()["__default__"].describe())

## Run

In [0]:
timings = run("__default__", spark, REPO, OVERRIDES, base_dir=os.getcwd(), from_node=FROM_NODE)
timings

✅ **Overall metrics**: compare our models with `reuniwatt_baseline`. Full report in `results/report_<product>.txt`.

In [0]:
S = dbutils.widgets.get("output_schema"); P = dbutils.widgets.get("product")
display(spark.table(f"{S}.uc2_metrics_summary_{P}"))